In [3]:
import torch

device = "cuda:0" if torch.cuda.is_available() else "cpu"

from progen3.modeling import ProGen3ForCausalLM
from progen3.batch_preparer import ProGen3BatchPreparer
from progen3.scorer import ProGen3Scorer

model = ProGen3ForCausalLM.from_pretrained("Profluent-Bio/progen3-112m", torch_dtype=torch.bfloat16)
model = model.eval().to("cuda:0")
batch_preparer = ProGen3BatchPreparer()

# Direct Usage
sequence = "MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN"

inputs = batch_preparer.get_batch_kwargs([sequence], device="cuda:0", reverse=False)
outputs = model(**inputs, return_dict=True)
print(outputs.logits)

# Usage with scorer (returns averaged log likelihood of forward and reverse direction)
# Would suggest using Scoring CLI below if scoring very large number of sequences
scorer = ProGen3Scorer(model=model)
scores = scorer.score_batch(sequences=[sequence])
print(scores["log_likelihood"][0])


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/225M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tensor([[[ -7.8438,  -7.8438,  -3.4375,  ...,  -7.8438,  -7.8438,  -7.8438],
         [-12.8750, -12.8750,  -8.1875,  ..., -12.8750, -12.9375, -12.8750],
         [-21.2500, -21.2500, -21.8750,  ..., -21.2500, -21.2500, -21.2500],
         ...,
         [-18.0000, -18.0000, -17.0000,  ..., -18.0000, -18.0000, -18.0000],
         [-10.0625, -10.0000,  36.7500,  ..., -10.1250, -10.0000, -10.0625],
         [-17.1250, -17.1250,   4.1875,  ..., -17.1250, -17.1250, -17.1250]]],
       device='cuda:0', grad_fn=<ToCopyBackward0>)
-2.7868616580963135


In [ ]:
import torch
from progen3.modeling import ProGen3ForCausalLM
from progen3.batch_preparer import ProGen3BatchPreparer

device = "cuda:0" if torch.cuda.is_available() else "cpu"

model = ProGen3ForCausalLM.from_pretrained("Profluent-Bio/progen3-112m",
                                           torch_dtype=torch.bfloat16).to(device)
model = model.eval()
batch_preparer = ProGen3BatchPreparer()

# Dummy sequence
sequence = "MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN"

# Prepare batch inputs
inputs = batch_preparer.get_batch_kwargs([sequence], device=device, reverse=False)

# Forward with hidden states
with torch.no_grad():
    outputs = model(**inputs, return_dict=True, output_hidden_states=True)

hidden_states = outputs.hidden_states  # tuple: length = num_layers+1

# Print number of layers
n_layers = model.config.num_hidden_layers
print(f"Model has {n_layers} transformer layers.")

print(f"hidden_states tuple length: {len(hidden_states)} (should be num_layers + 1)")

# Choose a layer (e.g., the middle)
layer_idx = n_layers // 2
h = hidden_states[layer_idx]  # shape: [batch_size, seq_len, hidden_dim]

print(f"Hidden state shape at layer {layer_idx}: {h.shape}")
print("Hidden vector for first token (truncated):", h[0, 0, :10])



Model has 10 transformer layers.
hidden_states tuple length: 11 (should be num_layers + 1)
Hidden state shape at layer 5: torch.Size([1, 114, 384])
Hidden vector for first token (truncated): tensor([ 17.1250,  -6.3125,  -7.1250,  12.8125, -19.3750,  -0.5703,   3.7500,
          1.3281,  -7.5000, -19.2500], device='cuda:0', dtype=torch.bfloat16)
